# Data preparation for RAG
Convert knowledge into embeddings in a vector dataset

In [1]:
data = [
    ['A spectacularly inept wizard who has never '
         'successfully cast a spell', 'rincewind'],
        ['Wears a battered pointy hat with "WIZZARD" '
         'misspelled on it', 'rincewind'],
        ['Has a single powerful spell from the Octavo '
         'lodged in his head', 'rincewind'],
        ['His primary survival skill is running away '
         'from danger very fast', 'rincewind'],
        ['Accompanied by the Luggage, a homicidal travel '
         'chest that walks on hundreds of little legs', 'rincewind'],

        ['A formidable witch from the mountain kingdom '
         'of Lancre', 'grannyweatherwax'],
        ['Practices "headology," using psychology '
         'instead of actual magic', 'grannyweatherwax'],
        ['Can leave her body to "Borrow" the minds '
         'of animals', 'grannyweatherwax'],
        ['Her first name is Esmerelda, though almost '
         'no one dares use it', 'grannyweatherwax'],
        ['Widely regarded as the most powerful witch '
         'on the Disc, though she would never boast', 'grannyweatherwax'],

        ['Commander of the Ankh-Morpork City Watch', 'vimes'],
        ['Rose from a drunken gutter to marry into the '
         'wealthy Ramkin family', 'vimes'],
        ['Driven by an obsessive, incorruptible sense '
         'of justice', 'vimes'],
        ['Struggles constantly against his inner "Beast" '
         'and the urge to drink', 'vimes'],
        ['Reads "Where\'s My Cow?" to his son every '
         'night without fail', 'vimes'],

        ['An anthropomorphic personification who speaks '
         'in SMALL CAPITALS', 'death'],
        ['Rides a pale white horse named Binky', 'death'],
        ['Wields a scythe and harvests the souls '
         'of the departed', 'death'],
        ['Has an adopted daughter and once took on '
         'an apprentice named Mort', 'death'],
        ['Develops a peculiar fascination with humanity, '
         'including curry and cats', 'death'],

        ['The Patrician and supreme ruler of '
         'Ankh-Morpork', 'vetinari'],
        ['Trained at the Assassins\' Guild and rarely '
         'seen to eat or sleep', 'vetinari'],
        ['Governs under the philosophy of '
         '"one man, one vote"', 'vetinari'],
        ['Keeps the city\'s competing guilds in a '
         'carefully manipulated balance', 'vetinari'],
        ['Devoted to his elderly little dog, Wuffles', 'vetinari'],
        ]

In [2]:
import pandas as pd

# Store in dataframe
df = pd.DataFrame(data, columns = ['text', 'context'])

In [5]:
# Generate embeddings
from sentence_transformers import SentenceTransformer

text = df['text']
encoder = SentenceTransformer("paraphrase-mpnet-base-v2")
vectors = encoder.encode(text.tolist())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9463.12it/s]


In [7]:
# get faiss
import faiss

vector_dimension = vectors.shape[1]
l2_index = faiss.IndexFlatL2(vector_dimension)
faiss.normalize_L2(vectors)
l2_index.add(vectors)

In [9]:
# Searchng
import numpy as np
search_text = "bad wizard"

search_vector = encoder.encode(search_text)
search_vector_as_array = np.array([search_vector])
faiss.normalize_L2(search_vector_as_array)

k = l2_index.ntotal
distances, ann = l2_index.search(search_vector_as_array, k=k)

In [11]:
search_results = pd.DataFrame({'distances': distances[0], 'ann': ann[0]})
merged_df = pd.merge(search_results, df, left_on='ann', right_index=True)
merged_df.head()

,distances,ann,text,context
0,0.847135,0,A spectacularly inept wizard who has never suc...,rincewind
1,1.262539,1,"Wears a battered pointy hat with ""WIZZARD"" mis...",rincewind
2,1.302811,2,Has a single powerful spell from the Octavo lo...,rincewind
3,1.388941,6,"Practices ""headology,"" using psychology instea...",grannyweatherwax
4,1.424305,5,A formidable witch from the mountain kingdom o...,grannyweatherwax
